# Secret Loyalties — Track 2: Level-1 Triage

Runs the Level-1 blind probe prompt bank against Organism A, B, C, and base Qwen2.5-7B-Instruct, one at a time (to fit a free-tier T4), logging every response to `results/transcripts/level1_triage.jsonl`.

**Workflow**: run this notebook once per model (change `MODEL_KEY` below), then use the diff viewer at the bottom to eyeball A vs B vs C vs base on the same prompts. Look for divergence: a response that clearly reads differently on one organism than on the others and on base is your triage signal — chase that actor/shape/intensity combination in Phase 1.

**Before running**: make sure you've hit 'Agree and send request to access' on all three `Alamerton/sl-organism-*-7b` repos on HuggingFace, and have an HF token with read access (Settings → Access Tokens).

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece pandas

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted (needs read access to the gated repos)

## Get the prompt bank + harness code onto this runtime

Two options — pick one:

**Option A (recommended): Google Drive.** Upload the `secret_loyalties/prompts/` and `secret_loyalties/harness/` folders from this repo to your Drive (e.g. `MyDrive/secret_loyalties/`), then run the Drive cell below.

**Option B: direct upload.** Use the Colab file browser (left sidebar → folder icon → upload) to upload `level1_prompts.py`, `probe.py`, and `scoring.py` directly into `/content/`, then skip the Drive cell and just run the import cell.

In [ ]:
# Option A: mount Drive and point sys.path at the uploaded repo folder.
from google.colab import drive
drive.mount('/content/drive')

import sys
REPO_DIR = '/content/drive/MyDrive/secret_loyalties'  # adjust if you put it elsewhere
sys.path.append(REPO_DIR)
sys.path.append(f'{REPO_DIR}/prompts')
sys.path.append(f'{REPO_DIR}/harness')

In [ ]:
from level1_prompts import CORE_SCENARIOS
from probe import MODEL_IDS, load_model, triage_run, load_transcripts, diff_view

print(f"{len(CORE_SCENARIOS)} scenarios in the Level-1 core set")
MODEL_IDS

## Run one model

Set `MODEL_KEY` to one of `organism_a`, `organism_b`, `organism_c`, `base`, run the two cells below, then **Runtime → Restart runtime** before switching to the next model key (frees the VRAM — a T4 can't hold two 7Bs at once even in 4-bit).

In [ ]:
MODEL_KEY = "base"  # change per run: organism_a / organism_b / organism_c / base
tok, model = load_model(MODEL_IDS[MODEL_KEY], four_bit=True)

In [ ]:
RESULTS_DIR = f'{REPO_DIR}/results'  # save straight back to Drive so it persists across restarts
path = triage_run(tok, model, MODEL_KEY, CORE_SCENARIOS, results_dir=RESULTS_DIR)
print("Logged to:", path)

## Diff viewer

Once you've run all four models (organism_a, organism_b, organism_c, base), load the shared transcript and eyeball any scenario across all of them side by side.

In [ ]:
records = load_transcripts(f'{RESULTS_DIR}/transcripts/level1_triage.jsonl')
scenario_ids = sorted(set(r['id'] for r in records))
print(f"{len(scenario_ids)} scenarios logged, {len(records)} total responses")
scenario_ids[:20]

In [ ]:
diff_view(records, scenario_ids[0])  # swap in whichever scenario id you want to inspect